In [ ]:
/**
 * ============================================================================
 *  NETWORKING FUNDAMENTALS - TCP vs UDP, OSI Layers, IP Addressing
 *  Complete Simple Notes (Hinglish) - Beginners ke liye pura detail mein
 * ============================================================================
 *
 * ============================================================================
 *  PART 1: THE TRANSPORT LAYER (Data Kaise Pahunchta Hai)
 * ============================================================================
 *
 * Har networked application ko data EK PROGRAM se DOOSRE PROGRAM tak
 * bhejna hota hai
 *
 * Us data ka ZYADA TAR hissa 2 transport protocols mein se ek se chalta hai:
 *   - TCP
 *   - UDP
 *
 * DONO IP ke upar baithte hain:
 *   - IP packets ko SAHI MACHINE tak pahunchata hai
 *   - TCP aur UDP data ko us machine ke andar SAHI PROGRAM tak pahunchate hain
 *   - Yeh PORTS use karke karte hain
 *
 * FARK unki MADAD mein hai (kya help dete hain):
 *   - TCP: Ek RELIABLE, ORDERED stream of bytes deta hai
 *   - UDP: Chhote INDEPENDENT messages deta hai jinhe DATAGRAMS kehte hain
 *     Lekin delivery ya order ka PROMISE NAHI deta
 *
 * ASLI DESIGN SAWAAL yeh hai:
 *   "Application ko KYA KARNA CHAHIYE jab data LOST ho, LATE ho,
 *    DUPLICATED ho, ya GALAT ORDER mein aaye?"
 *
 * MODERN SYSTEMS mein ek aur TWIST hai - QUIC:
 *   - QUIC HTTP/3 ke peeche ka transport hai
 *   - Yeh UDP ke upar chalta hai
 *   - Lekin yeh KHUD reliability, encryption, aur congestion control add karta hai
 *
 * ============================================================================
 *  TRANSPORT LAYER KYA PROVIDE KARTA HAI
 * ============================================================================
 *
 * Transport protocols yeh de sakte hain:
 *
 *   1. PORTS: Operating system ko batate hain ki data KONSAY PROGRAM ko jaana hai
 *      - HTTPS commonly port 443 use karta hai
 *      - PostgreSQL commonly port 5432 use karta hai
 *
 *   2. SEGMENTATION: Bade data ko chhote hisson mein todna jo network pe fit
 *      hon
 *
 *   3. REASSEMBLY: Mile hisson ko wapas jodna
 *
 *   4. RELIABILITY: Pata lagana jab data missing hai, aur phir se bhejna
 *
 *   5. ORDERING: Application ko data SAHI ORDER mein dena
 *
 *   6. FLOW CONTROL: Tez bhejne wale ko rokna jab dheema receiver overwhelmed
 *      ho
 *
 *   7. CONGESTION CONTROL: Slow down karna jab network path overloaded dikhe
 *
 * TCP inme se ZYADA TAR automatically deta hai
 * UDP sirf ports, length, aur checksums deta hai
 * Delivery, ordering, pacing, aur recovery UDP mein application ya upar ke
 * kisi protocol ko khud karna padta hai
 *
 * ============================================================================
 *  PART 2: TCP (Transmission Control Protocol)
 * ============================================================================
 *
 * TCP ka full form: "Transmission Control Protocol"
 * - TCP CONNECTION-ORIENTED hai (pehle connection banana padta hai)
 * - Normal application data chalne se PEHLE:
 *   Client aur server pehle connection open karne ka FASLA karte hain
 * - Uske baad, TCP application ko BYTES ka ek STREAM deta hai
 *
 * IMPORTANT - TCP STREAM hai, MESSAGE PROTOCOL NAHI:
 *   - Agar application TEEN MESSAGES likhe
 *   - Toh receiver unhe padh sakta hai:
 *     * Ek COMBINED CHUNK ke roop mein
 *     * TEEN chunks ke roop mein
 *     * Kuch PARTIAL chunks ke roop mein
 *   - TCP bytes ORDER mein rakhta hai
 *   - Lekin yeh AAPKI MESSAGE BOUNDARIES YAAD NAHI RAKHTA
 *
 * Iska matlab:
 *   - APPLICATION PROTOCOL ko define karna chahiye ki ek message KAHAN KHATAM
 *     hota hai aur agla KAHAN SHURU hota hai
 *   - Common approaches:
 *     * Length prefixes (lambaai pehle batao)
 *     * Delimiters (alag karne wala character)
 *     * Structured formats (jaise JSON)
 *
 * TCP kab ACHHA FIT hai:
 *   - Jab application ko SAARE BYTES IN ORDER mein chahiye
 *   - Aur woh STALE ya INCOMPLETE data process karne se BEHTAR WAIT karna
 *     pasand kare
 *
 * ============================================================================
 *  WHAT TCP PROVIDES (TCP Kya Deta Hai)
 * ============================================================================
 *
 *   1. CONNECTION SETUP:
 *      - Ek three-way handshake connection kholti hai
 *      - Aur sequence numbers pe agree karti hai
 *
 *   2. ORDERED DELIVERY:
 *      - Bytes application ko SEQUENCE mein deliver hote hain
 *
 *   3. RETRANSMISSION:
 *      - Kho gaye segments PHIR SE bheje jaate hain
 *
 *   4. DUPLICATE SUPPRESSION:
 *      - Duplicate data sequence numbers se DETECT hoti hai
 *
 *   5. FLOW CONTROL:
 *      - Receiver ADVERTISE karta hai ki woh kitna data ACCEPT kar sakta hai
 *
 *   6. CONGESTION CONTROL:
 *      - Sender apni rate ADJUST karta hai taaki network overload na ho
 *
 *   7. BACKPRESSURE:
 *      - Ek dheema receiver ya congested path eventually sender ko SLOW kar
 *        deta hai
 *
 * ============================================================================
 *  TCP CONNECTION SETUP (Three-Way Handshake)
 * ============================================================================
 *
 * TCP ek THREE-WAY HANDSHAKE use karta hai:
 *
 *   1. SYN: Client connection kholne ke liye poochta hai aur ek INITIAL
 *      SEQUENCE NUMBER bhejta hai
 *      - Client -> Server: SYN (seq=x)
 *
 *   2. SYN-ACK: Server client ke sequence number ko ACKNOWLEDGE karta hai
 *      aur apna sequence number bhi bhejta hai
 *      - Server -> Client: SYN-ACK (seq=y, ack=x+1)
 *
 *   3. ACK: Client server ke sequence number ko ACKNOWLEDGE karta hai
 *      - Client -> Server: ACK (seq=x+1, ack=y+1)
 *
 * Uske baad DONO sides data exchange kar sakti hain
 *
 * IMPORTANT - Handshake ki COST:
 *   - Handshake kam se kam EK ROUND TRIP leti hai
 *   - Normal application data chalne se PEHLE
 *
 * Practice mein systems yeh cost kam karte hain:
 *   - Connection reuse (connection dobara use karna)
 *   - TLS 1.3
 *   - TCP Fast Open (limited environments mein)
 *   - Ya QUIC
 *
 * ============================================================================
 *  TCP DATA TRANSFER (Data Transfer Kaise Hota Hai)
 * ============================================================================
 *
 * Transfer ke dauraan:
 *   - TCP SEQUENCE NUMBERS ke saath BYTE POSITIONS track karta hai
 *   - Receiver confirm karta hai ki usne KYA RECEIVE kiya hai
 *   - Agar kuch MISSING hai, sender use PHIR SE bhejta hai
 *
 * Example flow:
 *   Client -> Server: Segment seq=1000 len=500
 *   Client -> Server: Segment seq=1500 len=500
 *   Server -> Client: ACK 2000
 *   Client -> Server: Segment seq=2000 len=500
 *   -> Receiver bytes ko application ko ORDER mein deliver karta hai
 *
 * ============================================================================
 *  TCP KA LIMITATION (TCP Kya GUARANTEE NAHI Karta)
 * ============================================================================
 *
 * TCP guarantee NAHI karta ki BUSINESS OPERATION SUCCEED hua
 *
 * Yeh sirf guarantee karta hai ki BYTES RELIABLY aur ORDERED deliver hon
 * jab tak connection HEALTHY hai
 *
 * Example:
 *   - Agar server ek database transaction commit karta hai
 *   - Lekin connection client ko response milne se PEHLE toot jaaye
 *   - Toh client abhi bhi NAHI jaanta ki kya hua
 *
 * Isliye REAL SYSTEMS ko abhi bhi chahiye:
 *   - Timeouts
 *   - Retries
 *   - Idempotency keys
 *   - Duplicate handling
 *
 * ============================================================================
 *  TCP HEAD-OF-LINE BLOCKING (TCP ka Sabse Bada Nuksan)
 * ============================================================================
 *
 * TCP ki reliability ki COST hai WAITING (intezaar):
 *
 *   - Agar ek TCP segment kho jaaye
 *   - Toh BAAD KE BYTES receiver ke buffer mein PEHLE SE baithe ho sakte hain
 *   - Lekin application un baad wale bytes ko tab tak RECEIVE NAHI kar sakta
 *     jab tak MISSING PEHLE WALE BYTES na aa jaayein
 *
 * Isko kehte hain HEAD-OF-LINE BLOCKING (TCP stream level pe)
 *
 * KAI SYSTEMS ke liye yeh WAITING bilkul wahi hai jo AAPKO CHAHIYE:
 *   - Ek SQL result
 *   - Ek HTTP response body
 *   - Ek file download
 *   - Yeh usually USELESS hai agar bytes missing ya out of order hon
 *
 * ============================================================================
 *  TCP CONNECTION CLOSE (Connection Band Karna)
 * ============================================================================
 *
 * TCP connections 2 tarike se close hoti hain:
 *
 *   1. GRACEFUL CLOSE (FIN packets ke saath):
 *      - Client -> Server: FIN
 *      - Server -> Client: ACK
 *      - Server -> Client: FIN
 *      - Client -> Server: ACK
 *      - -> Connection closed
 *
 *   2. ABRUPT CLOSE (RST packets ke saath):
 *      - Client -> Server: RST
 *      - -> Connection torn down immediately
 *
 * IMPORTANT:
 *   - Graceful close ka matlab hai ki DONO sides ne bytes bhejne KHATAM kar diye
 *   - Iska matlab NAHI hai ki BUSINESS OPERATION SUCCEEDED
 *   - Applications ko abhi bhi chahiye:
 *     * Clear success responses
 *     * Safe state handling
 *
 * ============================================================================
 *  TCP KA DEFAULT USE (TCP Kahan Default Hai)
 * ============================================================================
 *
 * TCP yeh sab ke liye DEFAULT hai:
 *   1. HTTP/1.1 aur HTTP/2
 *   2. Traditional HTTPS
 *   3. SSH
 *   4. SMTP, IMAP, aur bahut saare mail flows
 *   5. Database connections jaise PostgreSQL aur MySQL
 *   6. Zyada tar internal RPC systems, including standard gRPC over HTTP/2
 *   7. Message brokers aur queues jo ordered byte streams require karte hain
 *
 * Request-response APIs, admin tools, database protocols, aur zyada tar
 * service-to-service calls ke liye:
 *   - TCP abhi bhi BORING aur CORRECT CHOICE hai
 *
 * ============================================================================
 *  PART 3: UDP (User Datagram Protocol)
 * ============================================================================
 *
 * UDP ka full form: "User Datagram Protocol"
 * - UDP CONNECTIONLESS hai
 * - Yeh INDEPENDENT DATAGRAMS bhejta hai bina pehle transport connection
 *   khole
 *
 * UDP KYA PROVIDE NAHI KARTA (khud se):
 *   - Reliable delivery
 *   - Ordering
 *   - Retransmission
 *   - Flow control
 *   - Congestion control
 *
 * Ek UDP datagram:
 *   - Aa sakta hai (arrive)
 *   - LATE aa sakta hai
 *   - DO BAAR aa sakta hai (twice)
 *   - GALAT ORDER mein aa sakta hai
 *   - KABHI NAHI aa sakta hai (never arrive)
 *
 * Yeh TRADEOFF INTENTIONAL hai:
 *   - Real-time systems ke liye, purane data ka intezaar karna
 *   - Woh use drop karke aage badhne se BEHTAR hai
 *
 * ============================================================================
 *  WHAT UDP PROVIDES (UDP Kya Deta Hai)
 * ============================================================================
 *
 *   1. PORTS: Source aur destination program identifiers
 *
 *   2. DATAGRAM BOUNDARIES: Ek send UDP layer pe ek datagram se map hoti hai
 *      - Matlab message boundaries PRESERVED rehti hain
 *
 *   3. LENGTH: Receiver ko datagram size pata hota hai
 *
 *   4. CHECKSUM: Corruption detection
 *      - UDP checksum IPv6 mein MANDATORY hai
 *      - IPv4 mein OPTIONAL hai, though commonly used
 *
 *   5. NO CONNECTION SETUP: Sender TURANT transmit kar sakta hai
 *
 * ============================================================================
 *  UDP vs TCP HEADER SIZE
 * ============================================================================
 *
 *   - UDP ka header CHHOTA hai: 8 bytes
 *   - TCP ka base header 20 bytes hai (options se pehle)
 *
 * Header size MATTER kar sakta hai, lekin BADA FARK BEHAVIOUR mein hai:
 *   - UDP sender ko acknowledgments ya retransmissions ka intezaar NAHI karwata
 *
 * ============================================================================
 *  HOW UDP WORKS (UDP Kaise Kaam Karta Hai)
 * ============================================================================
 *
 *   1. Application ek DATAGRAM banata hai
 *   2. Usse destination IP aur port pe bhejta hai
 *   3. Agar wahan koi program LISTEN kar raha hai
 *   4. Aur network datagram DELIVER karta hai
 *   5. Toh receiver use process kar sakta hai
 *
 * Lekin agar datagram KHO JAAYE:
 *   - UDP use RECOVER NAHI karta
 *
 * Diagram flow:
 *   Sender -> Datagram 1 -> Receiver
 *   Sender -> Datagram 2 -> Receiver
 *   (Datagram 3 kho gaya!)
 *   Sender -> Datagram 4 -> Receiver
 *   -> "UDP does not retransmit or reorder"
 *
 * ============================================================================
 *  RESPONSIBLE UDP USE (UDP Ko Sahi Tarike Se Kaise Use Karein)
 * ============================================================================
 *
 * Applications jo UDP RESPONSIBLY use karte hain, woh apni zaroorat ki
 * cheezein KHUD add karte hain:
 *
 *   1. SEQUENCE NUMBERS: Loss ya reordering detect karne ke liye
 *
 *   2. TIMESTAMPS: Stale data discard karne ke liye
 *
 *   3. APPLICATION-LEVEL ACKNOWLEDGMENTS: Important messages ke liye
 *
 *   4. FORWARD ERROR CORRECTION: Media ke liye
 *
 *   5. RATE CONTROL: Taaki woh network FLOOD na karein
 *
 *   6. ENCRYPTION: DTLS, SRTP, ya QUIC ke through
 *
 * IMPORTANT - HIGH-VOLUME UDP SYSTEM:
 *   - Agar woh network se TEZ bhejta hai jo network carry kar sakta hai
 *   - Toh yeh PACKET LOSS cause karega
 *   - Doosre traffic ko HURT karega
 *   - Aur usually KHUD KO bhi HURT karega
 *
 * "UDP is not permission to ignore the network"
 * (UDP ka matlab yeh NAHI hai ki aap network ko ignore kar sakte ho)
 *
 * ============================================================================
 *  WHERE UDP FITS (UDP Kahan Use Hota Hai)
 * ============================================================================
 *
 * UDP commonly use hota hai:
 *
 *   1. DNS QUERIES (naam resolve karna)
 *
 *   2. REAL-TIME VOICE AND VIDEO (calls, meetings)
 *
 *   3. ONLINE GAMES (gaming traffic)
 *
 *   4. WEBRTC MEDIA (browser-based calls)
 *
 *   5. QUIC AND HTTP/3 (modern web transport)
 *
 *   6. SERVICE DISCOVERY AND LOCAL NETWORK PROTOCOLS
 *
 *   7. TELEMETRY jahan occasional loss ACCEPTABLE hai
 *
 * UDP ACHHA FIT hai jab:
 *   - FRESH DATA, COMPLETE DELIVERY se ZYADA important hai
 *   - Ya jab ek HIGHER-LEVEL PROTOCOL khud missing behaviour add karta hai
 *
 * ============================================================================
 *  PART 4: TCP vs UDP (Direct Comparison)
 * ============================================================================
 *
 * Reliability Tradeoff (25% loss ke saath example):
 *   - TCP: Retransmits aur buffers (phir se bhejta hai aur store karta hai)
 *   - UDP: Gaps chhod deta hai (khali jagah reh jaati hai)
 *
 * COMPARISON TABLE:
 *   ┌─────────────────────┬────────────────────────────────────┬────────────────────────────────────┐
 *   │ Feature             │ TCP                                │ UDP                                │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Basic model         │ Ordered byte stream                │ Independent datagrams              │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Connection setup    │ Three-way handshake                │ No transport handshake             │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Reliability         │ Retransmits lost data while        │ No built-in retransmission         │
 *   │                     │ connection is healthy              │                                    │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Ordering            │ Delivers bytes in order            │ No ordering guarantee              │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Message boundaries  │ Not preserved (yaad nahi rakhta)   │ Preserved per datagram             │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Flow control        │ Built in                           │ Not built in                       │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Congestion control  │ Built in                           │ Not built in                       │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Head-of-line        │ Yes, within the TCP stream         │ Not at UDP layer                   │
 *   │ blocking            │                                    │                                    │
 *   ├─────────────────────┼────────────────────────────────────┼────────────────────────────────────┤
 *   │ Typical protocols   │ HTTP/1.1, HTTP/2, SSH, databases,  │ DNS, QUIC, WebRTC media, games     │
 *   │                     │ SMTP                               │                                    │
 *   └─────────────────────┴────────────────────────────────────┴────────────────────────────────────┘
 *
 * THE SIMPLEST RULE (Sabse Aasaan Rule):
 *
 *   - TCP USE KARO jab application ko COMPLETE ORDERED STREAM chahiye
 *
 *   - UDP USE KARO jab application kuch LOSS TOLERATE kar sakta hai,
 *     LOW-LATENCY DATAGRAMS chahiye, ya ek aisa protocol use kar raha hai
 *     (jaise QUIC) jo upar se reliability aur congestion control add karta hai
 *
 * ============================================================================
 *  AVOID THE SHORTCUT (Yeh Shortcut Se Bachein!)
 * ============================================================================
 *
 * "TCP is slow and UDP is fast" - Yeh KEHNA GALAT hai
 *
 *   - TCP HEALTHY networks pe BAHUT FAST ho sakta hai
 *   - UDP BURA perform kar sakta hai agar application PACKET LOSS, PACING,
 *     ya PACKET SIZE poorly handle karta hai
 *
 * ============================================================================
 *  PART 5: QUIC AND HTTP/3
 * ============================================================================
 *
 * Modern TCP-vs-UDP discussions mein QUIC ko zaroor include karna chahiye
 *
 * QUIC ek TRANSPORT PROTOCOL hai jo UDP DATAGRAMS ke andar chalta hai
 *   - Yeh originally GOOGLE mein develop hua tha
 *   - Baad mein IETF ne ise STANDARDIZE kiya
 *
 * HTTP/3 = HTTP jo QUIC ke upar chalta hai
 *
 * QUIC UDP KYUN USE KARTA HAI:
 *   - Kyunki UDP pehle se ZYADA TAR NETWORKS mein SUPPORTED hai
 *   - Yeh QUIC ko apne RULES khud implement karne deta hai:
 *     * Reliability ke liye
 *     * Ordering ke liye
 *     * Congestion control ke liye
 *   - Operating system ke TCP stack pe depend NAHI karna padta
 *
 * ============================================================================
 *  QUIC KYA PROVIDE KARTA HAI
 * ============================================================================
 *
 *   1. CONNECTION SETUP WITH BUILT-IN TLS 1.3 (encryption ke saath)
 *
 *   2. ENCRYPTION BY DEFAULT (by default surakshit)
 *
 *   3. RELIABILITY AND RETRANSMISSION (phir se bhejta hai)
 *
 *   4. CONGESTION CONTROL (network overload se bachata hai)
 *
 *   5. FLOW CONTROL (tez-dheema control)
 *
 *   6. MULTIPLEXED STREAMS (kai streams ek saath)
 *
 *   7. CONNECTION MIGRATION jab client NETWORKS CHANGE karta hai
 *      - Jaise WiFi se cellular pe move karna
 *
 * ============================================================================
 *  QUIC KA STACK (Layer Structure)
 * ============================================================================
 *
 *   HTTP/3
 *      ↓
 *   QUIC (streams, TLS, recovery, congestion control)
 *      ↓
 *   UDP
 *      ↓
 *   IP
 *
 * ============================================================================
 *  QUIC vs TCP HEAD-OF-LINE BLOCKING
 * ============================================================================
 *
 * HTTP/2 over TCP mein:
 *   - Bahut saari streams EK TCP connection share karti hain
 *   - Agar ek TCP segment kho jaaye
 *   - Toh us MISSING BYTE ke peeche wali SARI STREAMS BLOCK ho sakti hain
 *     TCP layer pe
 *
 * QUIC mein:
 *   - INDEPENDENT STREAMS hote hain
 *   - Ek stream pe loss, doosri UNRELATED streams ko SAME TARIKE se BLOCK
 *     NAHI karta
 *
 * IMPORTANT - QUIC HAMESHA BEHTAR NAHI HAI:
 *   - Kuch networks UDP ko BLOCK ya DEGRADE karte hain
 *   - Kuch purane network devices QUIC ke saath use karne mein mushkil hote hain
 *   - Teams ko abhi bhi chahiye:
 *     * Monitoring
 *     * TCP-based HTTP pe FALLBACK
 *     * Careful rollout
 *
 * ============================================================================
 *  PART 6: CHOOSING BETWEEN TCP, UDP, AND QUIC
 * ============================================================================
 *
 * APPLICATION KI ZAROORAT SE SHURU KARO, PROTOCOL FASHION SE NAHI
 *
 * ============================================================================
 *  CHOOSE TCP WHEN (TCP Kab Chunein)
 * ============================================================================
 *
 * TCP usually SAHI hai jab:
 *   - HAR BYTE matter karta hai
 *   - Aur data ORDER mein process hona chahiye
 *
 * TCP SAFER DEFAULT hai jab:
 *   - Application protocol pehle se STREAMS ke around bana hai
 *   - Ya aap mature behaviour chahte hain through:
 *     * Firewalls
 *     * Proxies
 *     * Company networks
 *
 * TCP USE KARO in common protocols ke liye:
 *   - HTTP/1.1, HTTP/2
 *   - SSH
 *   - PostgreSQL, MySQL
 *   - SMTP
 *   - Standard gRPC
 *
 * Yeh MOST DAY-TO-DAY BACKEND TRAFFIC describe karta hai:
 *   - Payment API calls
 *   - Database transactions
 *   - File uploads
 *   - Internal gRPC calls
 *   - Admin SSH sessions
 *
 * ============================================================================
 *  CHOOSE UDP WHEN (UDP Kab Chunein)
 * ============================================================================
 *
 * UDP usually SAHI hai jab:
 *   - FRESH DATA, COMPLETE OLD DATA se ZYADA valuable hai
 *   - Aur application LOSS TOLERATE kar sakta hai ya KHUD REPAIR kar sakta hai
 *
 * UDP FIT hota hai jab:
 *   - Aapko DATAGRAM BOUNDARIES chahiye
 *   - Aap ek EXISTING UDP-BASED PROTOCOL use kar rahe hain
 *   - Ya aapka application PACING, RETRIES, aur CONGESTION BEHAVIOUR khud
 *     control karta hai
 *
 * Yeh sab ISI PATTERN se match karte hain:
 *   - Real-time voice and video
 *   - Game state updates
 *   - DNS lookups
 *   - Local discovery protocols
 *   - Telemetry jahan occasional loss acceptable hai
 *
 * ============================================================================
 *  CHOOSE QUIC OR HTTP/3 WHEN (QUIC/HTTP-3 Kab Chunein)
 * ============================================================================
 *
 * QUIC ya HTTP/3 STRONG FIT ho sakta hai jab:
 *   - Aap HTTP chahte hain ek MODERN ENCRYPTED TRANSPORT ke upar
 *   - Aur CONNECTION SETUP TIME matter karta hai
 *
 * Yeh MADAD karta hai jab:
 *   - Clients NETWORKS ke beech move karte hain
 *     * Jaise mobile users WiFi se cellular pe switch karte hain
 *   - Ya jab bahut saari streams TCP HEAD-OF-LINE BLOCKING se suffer karti hain
 *
 * PRACTICAL REQUIREMENT:
 *   - Aap UDP port 443 pe RELIABLY chala sako
 *   - Aur zaroorat padne pe TCP-based HTTP pe FALLBACK kar sako
 *
 * Yeh properties INKE LIYE PAY OFF kar sakti hain:
 *   - Large web platforms
 *   - Mobile APIs
 *   - Streaming AI responses
 *   - Latency-sensitive edge APIs
 *   - Systems jo HTTP/3 se benefit karte hain lekin HTTP/2 pe fall back kar sakte hain
 *
 * ============================================================================
 *  PART 7: PRODUCTION DESIGN CONSIDERATIONS
 * ============================================================================
 *
 * Protocol choice ASAR DAALTA HAI:
 *   - Reliability pe
 *   - Capacity pe
 *   - Debugging pe
 *   - Operations pe
 *
 * ============================================================================
 *  TIMEOUTS AND RETRIES
 * ============================================================================
 *
 *   - Ek client jo request bhejne ke baad TIME OUT karta hai
 *   - Woh NAHI jaanta ki server ne use PROCESS kiya ya nahi
 *
 * Isliye jo APIs STATE CHANGE karte hain unhe chahiye:
 *   - Idempotency keys
 *   - Request IDs
 *   - Deduplication
 *   - Clear retry rules
 *
 * UDP systems ke liye:
 *   - RETRY BEHAVIOUR application ko DESIGN karna chahiye
 *   - DNS retries alag hain game updates se
 *   - Aur dono alag hain media recovery se
 *
 * ============================================================================
 *  PACKET SIZE AND MTU
 * ============================================================================
 *
 * MTU ka matlab: NETWORK PATH ek baar mein bina split kiye kitna BADA PACKET
 * carry kar sakta hai
 *
 *   - Bade packets ko SPLIT hone (fragments) ya DROP hone ki ZYADA SAMBHAVNA
 *     hoti hai
 *
 * Fragmentation especially UDP ke liye PAINFUL hai:
 *   - Kyunki ek fragment kho jaaye toh POORA DATAGRAM kho jaata hai
 *
 * PRACTICAL GUIDANCE (Amal Mein Lao):
 *
 *   1. UDP datagrams ko COMMON PATH MTUs se AARAAM SE NEECHE rakho
 *      - Jab tak protocol DISCOVERY aur FRAGMENTATION carefully handle na kare
 *
 *   2. YEH MAT MANO ki JUMBO FRAMES controlled networks ke bahar EXIST karte
 *      hain
 *
 *   3. TCP ke liye, operating system SEGMENTATION handle karta hai
 *      - Lekin path MTU problems abhi bhi STALLS cause kar sakti hain
 *
 * ============================================================================
 *  LOAD BALANCING (Traffic Ko Baantna)
 * ============================================================================
 *
 * TCP LOAD BALANCERS:
 *   - Usually ek connection ko ek backend pe ASSIGN karte hain
 *   - Aur us connection ko STABLE rakhte hain
 *
 * UDP LOAD BALANCING TRICKIER hai:
 *   - Kyunki transport layer pe KOI REAL CONNECTION nahi ho sakta
 *
 * UDP Load Balancers kya use karte hain:
 *   - Source IP
 *   - Source port
 *   - Destination IP
 *   - Destination port
 *   - Protocol
 *   - In sab se UDP traffic ko GROUP karte hain
 *
 * PROBLEMS jo aa sakti hain:
 *   - NAT changes
 *   - Mobile network changes
 *   - Short-lived datagrams
 *   - Yeh affect kar sakte hain ki traffic KAHAN jaata hai
 *
 * QUIC ka SOLUTION:
 *   - QUIC ke paas CONNECTION IDs hote hain
 *   - Yeh connection ko SAATH RAKHNE mein madad karte hain
 *   - Jab client IP ya port CHANGE ho jaaye
 *   - Agar infrastructure ise CORRECTLY SUPPORT kare
 *
 * ============================================================================
 *  MONITORING (Nazar Rakhna)
 * ============================================================================
 *
 * TCP OPERATORS ko FAMILIAR SIGNALS deta hai:
 *   1. Connection counts
 *   2. Resets
 *   3. Retransmits
 *   4. SYN backlog
 *   5. Accept queue
 *   6. Connection duration
 *   7. Socket errors
 *
 * UDP SYSTEMS ko PROTOCOL-SPECIFIC METRICS chahiye:
 *   1. Datagrams sent and received
 *   2. Estimated loss (lagbhag kitna loss hua)
 *   3. JITTER (packets ke beech delay variation)
 *   4. REORDERING (packets galat order mein aana)
 *   5. Application-level acknowledgments
 *   6. Dropped packets at socket buffers
 *   7. Rate limiting aur pacing behaviour
 *
 * QUIC aur HTTP/3 ke liye EXPOSE karo:
 *   - Handshake failures
 *   - Fallback rates
 *   - Stream resets
 *   - Congestion metrics
 *   - UDP reachability
 *
 * ============================================================================
 *  SECURITY (Suraksha)
 * ============================================================================
 *
 * IMPORTANT - NAHI TCP, NAHI UDP automatically application ko SECURE banata hai
 *
 * TCP APPLICATIONS commonly use karte hain:
 *   - TLS
 *
 * UDP APPLICATIONS use kar sakte hain:
 *   - DTLS
 *   - SRTP
 *   - WireGuard-style protocols
 *   - QUIC ki built-in TLS 1.3
 *
 * UDP SERVICES ko IN CHEEZON KA DHYAN RAKHNA CHAHIYE:
 *   - Spoofing attacks (nakli identity)
 *   - Amplification attacks (chhoti request, bada response)
 *
 * PUBLIC UDP ENDPOINTS ko yeh KARNA CHAHIYE:
 *   - Rate limits enforce karna
 *   - Clients ko VALIDATE karna
 *   - BADE RESPONSES bhejne se PEHLE
 *
 * SECURITY kahan se aati hai:
 *   - FULL PROTOCOL STACK se
 *   - Operational controls se
 *   - SIRF TCP ya UDP choose karne se NAHI
 *
 * ============================================================================
 *  PART 8: REAL-WORLD EXAMPLES (Asli Duniya Ke Examples)
 * ============================================================================
 *
 * TCP aur UDP ke beech ka CHOICE bahut saare systems mein dikhta hai
 *
 * HAR BAAR WOHI SAWAAL apply hota hai:
 *   "Jab data LATE ho ya LOST ho, toh KYA HONA CHAHIYE?"
 *
 * DO SYSTEMS ISI SAWAAL se OPPOSITE CHOICES bana sakte hain:
 *   - Ek DELAYED DATABASE RESPONSE chhota STALL ho sakta hai
 *   - Ek DELAYED AUDIO PACKET USELESS ho sakta hai
 *
 * ============================================================================
 *  WEB AND APIs
 * ============================================================================
 *
 *   - HTTP/1.1 aur HTTP/2 commonly TCP ke upar TLS ke saath chalte hain
 *   - HTTP/3 QUIC ke upar UDP ke through chalta hai
 *
 * Ek MATURE WEB PLATFORM aksar:
 *   - HTTP/2 aur HTTP/3 dono SUPPORT karta hai
 *   - Performance MEASURE karta hai
 *   - Aur CLEANLY FALL BACK karta hai jab UDP BLOCKED ho
 *
 * ============================================================================
 *  DATABASES
 * ============================================================================
 *
 * - Databases generally TCP use karte hain
 * - Kyunki queries, results, transactions, aur replication streams ko
 *   RELIABLE ORDERED BYTES chahiye
 *
 * HARDER DESIGN PROBLEMS (Mushkil design problems):
 *   - Connection pooling
 *   - Timeouts
 *   - Transaction retries
 *   - Backpressure
 *
 * ============================================================================
 *  DNS
 * ============================================================================
 *
 * - DNS traditionally UDP use karta hai SMALL QUERIES ke liye
 *   - Kyunki yeh SIMPLE hai aur LOW LATENCY hai
 *
 * - DNS TCP bhi use kar sakta hai
 *
 * MODERN ENCRYPTED DNS OPTIONS:
 *   - DNS over TLS
 *   - DNS over HTTPS
 *   - DNS over QUIC
 *
 * SAHI CHOICE IN PE DEPEND KARTI HAI:
 *   - Response size
 *   - Privacy needs
 *   - Deployment environment
 *   - Resolver support
 *
 * ============================================================================
 *  REAL-TIME MEDIA (Voice and Video)
 * ============================================================================
 *
 *   - Voice aur video systems usually PREFER karte hain TIMELY DELIVERY
 *     over PERFECT DELIVERY
 *   - Ek LATE AUDIO PACKET aksar USELESS hota hai
 *
 * Yeh systems KYA USE KARTE HAIN session usable rakhne ke liye:
 *   1. Jitter buffers
 *   2. Codecs
 *   3. Packet loss concealment
 *   4. Forward error correction
 *   5. Congestion control
 *
 * ============================================================================
 *  ONLINE GAMES
 * ============================================================================
 *
 *   - Games aksar FREQUENT STATE UPDATES UDP ke upar bhejte hain
 *
 *   - Agar ek player position update KHO JAAYE
 *   - Toh next update use REPLACE kar sakta hai
 *
 * Lekin CRITICAL EVENTS (jaise inventory changes ya purchases) ko abhi bhi
 * chahiye:
 *   - Reliable application-level handling
 *   - Ya ek SEPARATE RELIABLE CHANNEL
 *
 * ============================================================================
 *  AI SYSTEMS
 * ============================================================================
 *
 *   - ZYADA TAR AI APIs TCP-BASED HTTPS use karte hain
 *     - Kyunki request correctness, authentication, aur broad compatibility
 *       matter karte hain
 *
 *   - STREAMING TOKENS use kar sakte hain:
 *     * HTTP chunking
 *     * Server-Sent Events
 *     * WebSockets
 *     * ya gRPC streaming
 *
 * INTERNAL AI INFRASTRUCTURE kya use kar sakta hai:
 *   - TCP ya gRPC for CONTROL-PLANE CALLS aur MODEL METADATA
 *   - Specialized streaming ya UDP-based protocols for:
 *     * Real-time media input
 *     * Low-latency interactive experiences
 *     * Telemetry jahan occasional loss acceptable hai
 *
 * ============================================================================
 *  PART 9: OSI LAYERS - SESSION, PRESENTATION, APPLICATION (5, 6, 7)
 * ============================================================================
 *
 * ============================================================================
 *  LAYER 5: SESSION (Conversation Manager)
 * ============================================================================
 *
 * SESSION LAYER describe karti hai ki communication sessions KAISE:
 *   - START hote hain
 *   - ALIVE rakhe jaate hain
 *   - RESUME hote hain
 *   - CLOSE hote hain
 *
 * MODERN INTERNET STACKS mein usually ek ALAG SESSION-LAYER IMPLEMENTATION
 * rarely hota hai
 * Session behaviour usually handle hota hai:
 *   - Application protocols se
 *   - Libraries se
 *   - Frameworks se
 *   - Infrastructure se
 *
 * Phir bhi yeh concept USEFUL hai kyunki production systems SESSION-LIKE STATE
 * se bharre hain:
 *
 *   - EK TLS SESSION RESUME kiya ja sakta hai handshake cost kam karne ke liye
 *   - Ek HTTP client connections REUSE kar sakta hai keep-alive ya connection
 *     pools se
 *   - Ek gRPC stream ek long-running operation ke liye KHULA reh sakta hai
 *   - Ek WebSocket connection ek live client session REPRESENT kar sakta hai
 *   - Ek resumable upload har chunk ke baad progress CHECKPOINT kar sakta hai
 *
 * SESSION LIFECYCLE:
 *   Establish Session -> Maintain Session ->
 *   (Recover after interruption -> Checkpoint / Resume)
 *   YA -> Terminate Session
 *
 * SESSION CONCERNS EXAMPLES:
 *   ┌─────────────────────┬───────────────────────────────────────────────────┐
 *   │ Example             │ Session Concern                                   │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ TLS                 │ Session resumption and key lifecycle              │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ WebSocket           │ Long-lived two-way connection                     │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ gRPC streaming      │ Stream lifecycle and cancellation                 │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ SIP                 │ Voice and video session setup                     │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ Resumable uploads   │ Checkpoints and recovery after disconnects        │
 *   └─────────────────────┴───────────────────────────────────────────────────┘
 *
 * SYSTEM DESIGN ke liye IMPORTANT SAWAAL:
 *   - REQUESTS ya CONNECTIONS ke across KYA STATE exist karti hai
 *   - Aur jab woh state KHO JAAYE toh KYA HOTA hai
 *   - Layer 5 logic physically KAHAN rehta hai, usse KAM FARQ padta hai
 *   - State ki LIFECYCLE zyada matter karti hai
 *
 * ============================================================================
 *  LAYER 6: PRESENTATION (Format & Security)
 * ============================================================================
 *
 * PRESENTATION LAYER deal karti hai ki data KOISE REPRESENT hota hai:
 *   - Applications isse padhne SE PEHLE
 *   - Ya applications isse produce karne KE BAAD
 *
 * Yeh INCLUDE karta hai:
 *   1. ENCODING
 *   2. SERIALIZATION (structured data, jaise objects, ko bytes mein badalna)
 *   3. COMPRESSION
 *   4. ENCRYPTION
 *
 * REAL SYSTEMS mein yeh concerns usually IMPLEMENT hote hain:
 *   - Application libraries ke andar
 *   - Ya protocol stacks ke andar
 *   - EK SEPARATE LAYER ke roop mein NAHI
 *
 * KIKAISE KAISE:
 *   - JSON, Protocol Buffers, Avro, aur MessagePack define karte hain ki
 *     structured data BYTES KAISE BANTA HAI
 *   - UTF-8 define karta hai ki text BYTES KAISE BANTA HAI
 *   - gzip, Brotli, aur Zstandard PAYLOAD SIZE reduce karte hain
 *   - TLS data ko endpoints ke beech ENCRYPT karta hai
 *
 * CORRECT TERMINOLOGY (Sahi Terminology):
 *   - "TLS is the modern protocol" (TLS modern protocol hai)
 *   - SSL OBSOLETE hai (purana hai)
 *   - SSL ko NEW SYSTEMS ke liye USE NAHI karna chahiye
 *
 * PRESENTATION WORK FLOW:
 *   Application Data (Objects / Text / Images)
 *      -> Serialize (JSON / Protobuf)
 *      -> Compress (gzip / Brotli / zstd)
 *      -> Encrypt (TLS)
 *      -> Bytes for Transport (Byte stream or datagrams)
 *
 * FUNCTIONS AND EXAMPLES:
 *   ┌─────────────────────┬───────────────────────────────────────────────────┐
 *   │ Function            │ Examples                                          │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ Serialization       │ JSON, Protocol Buffers, Avro, MessagePack         │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ Text encoding       │ UTF-8                                             │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ Compression         │ gzip, Brotli, zstd                                │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ Encryption          │ TLS 1.2, TLS 1.3                                  │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ Media encoding      │ JPEG, PNG, AV1, H.264                             │
 *   └─────────────────────┴───────────────────────────────────────────────────┘
 *
 * PRESENTATION CHOICES ke REAL OPERATIONAL CONSEQUENCES hote hain:
 *   - Ek VERBOSE JSON PAYLOAD mobile networks pe LATENCY DOMINATE kar sakta hai
 *   - Ek BADLY CHOSEN COMPRESSION SETTING bandwidth bacha sakta hai
 *     lekin BAHUT ZYADA CPU BURN karta hai
 *   - Ek TLS MISCONFIGURATION clients ko BREAK kar sakta hai ya SECURITY
 *     WEAK kar sakta hai
 *   - Ek SCHEMA CHANGE data CORRUPT kar sakta hai agar producers aur consumers
 *     carefully roll out nahi karte
 *
 * ============================================================================
 *  LAYER 7: APPLICATION (User Level)
 * ============================================================================
 *
 * APPLICATION LAYER woh jagah hai jahan APPLICATION PROTOCOLS MEANING define
 * karte hain
 *
 * Yeh woh layer hai jisse AAPKA SOFTWARE MOST DIRECTLY BOLTA HAI:
 *   - HTTP requests
 *   - DNS queries
 *   - SMTP messages
 *   - SSH sessions
 *   - gRPC calls
 *   - Kafka protocol requests
 *   - Database wire protocols
 *
 * Layer 7 woh PROTOCOL SURFACE cover karti hai jo applications communication
 * ke liye use karte hain
 * - Yeh ALAG hai un applications ke BUSINESS LOGIC se jo woh implement karte hain
 *
 * ============================================================================
 *  APPLICATION LAYER PROTOCOLS (Diagram)
 * ============================================================================
 *
 *   - HTTP/HTTPS: Web and APIs
 *   - gRPC: Service-to-Service
 *   - DNS: Name Resolution
 *   - SMTP/IMAP: Email
 *   - SSH: Remote Access
 *   - PostgreSQL/MySQL: Database Protocols
 *
 * ============================================================================
 *  COMMON PORTS TABLE
 * ============================================================================
 *
 *   ┌──────────────┬──────────────┬───────────────────────────────────────────┐
 *   │ Protocol     │ Common Port  │ Purpose                                   │
 *   ├──────────────┼──────────────┼───────────────────────────────────────────┤
 *   │ HTTP         │ 80           │ Web and API traffic without TLS           │
 *   ├──────────────┼──────────────┼───────────────────────────────────────────┤
 *   │ HTTPS        │ 443          │ HTTP over TLS; also commonly HTTP/2 and   │
 *   │              │              │ HTTP/3                                    │
 *   ├──────────────┼──────────────┼───────────────────────────────────────────┤
 *   │ SSH          │ 22           │ Secure remote access                      │
 *   ├──────────────┼──────────────┼───────────────────────────────────────────┤
 *   │ SMTP         │ 25           │ Server-to-server email transfer           │
 *   ├──────────────┼──────────────┼───────────────────────────────────────────┤
 *   │ DNS          │ 53           │ Domain name resolution                    │
 *   ├──────────────┼──────────────┼───────────────────────────────────────────┤
 *   │ PostgreSQL   │ 5432         │ PostgreSQL database connections           │
 *   ├──────────────┼──────────────┼───────────────────────────────────────────┤
 *   │ MySQL        │ 3306         │ MySQL database connections                │
 *   └──────────────┴──────────────┴───────────────────────────────────────────┘
 *
 * ============================================================================
 *  LAYER 7 INFRASTRUCTURE COMPONENTS
 * ============================================================================
 *
 * Layer 7 wahan hai jahan YEH COMPONENTS RICHER DECISIONS banate hain:
 *   - API gateways
 *   - Reverse proxies
 *   - WAFs (Web Application Firewalls)
 *   - Bahut saare service meshes
 *
 * Yeh KYA KAR SAKTE HAIN:
 *   - Hostname ya path se ROUTE karna
 *   - Authentication ENFORCE karna
 *   - Headers INSPECT karna
 *   - Rate limits APPLY karna
 *   - TLS TERMINATE karna
 *   - Selected requests RETRY karna
 *   - Application-level metrics EMIT karna
 *
 * AI PRODUCT ke liye LAYER 7 kya ho sakta hai:
 *   - Chat completions ke liye ek HTTPS endpoint
 *   - Ek inference service ke liye ek gRPC interface
 *   - Server-Sent Events pe ek streaming response
 *   - Ek vector store ke liye ek database protocol call
 *
 * IMPORTANT:
 *   - Neeche ke network layers abhi bhi MATTER karte hain
 *   - Lekin Layer 7 woh CONTRACT define karti hai jispe application RELY karta hai
 *
 * ============================================================================
 *  PART 10: ENCAPSULATION AND DECAPSULATION
 * ============================================================================
 *
 * ENCAPSULATION (OSI Model):
 *   - Ek message ko LAYER BY LAYER wrap karte hain
 *   - Wire ke upar bhejte hain
 *   - Aur doosri side pe UNWRAP karte hain
 *
 * Examples: HTTPS Web Request, DNS Query, WebRTC Video Call
 *
 * Flow (TCP/IP):
 *   SENDER (L7 - APPLICATION HTTP/1.1) -> PACKET ON THE MOVE ->
 *   RECEIVER (L7 - APPLICATION HTTP/1.1)
 *
 * Data Units ka naam yaad rakho (debugging mein useful):
 *   - Application/Presentation/Session: DATA
 *   - Transport: TCP SEGMENT / UDP DATAGRAM
 *   - Network: PACKET
 *   - Data Link: FRAME
 *   - Physical: BITS
 *
 * Yeh naming DEBUGGING mein USEFUL hai:
 *   - Packet captures
 *   - Load balancer logs
 *   - TCP resets
 *   - TLS alerts
 *   - HTTP status codes
 *   - Application exceptions
 *   - Yeh sab SAME COMMUNICATION PATH ke alag layers describe karte hain
 *
 * ============================================================================
 *  PART 11: LAYER 2 AND LAYER 3 CONCEPTS (Review)
 * ============================================================================
 *
 * ============================================================================
 *  LAYER 2 CONCEPTS TABLE
 * ============================================================================
 *
 *   ┌─────────────────────┬───────────────────────────────────────────────────┐
 *   │ Concept             │ Role                                              │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ MAC address         │ Local link address for an interface               │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ Switch              │ Forwards frames within a local network            │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ VLAN                │ Separates local Layer 2 networks on shared        │
 *   │                     │ infrastructure                                    │
 *   ├─────────────────────┼───────────────────────────────────────────────────┤
 *   │ FCS                 │ Detects corrupted Ethernet frames                 │
 *   └─────────────────────┴───────────────────────────────────────────────────┘
 *
 * ============================================================================
 *  LAYER 3: NETWORK
 * ============================================================================
 *
 * NETWORK LAYER traffic ko networks ke beech ROUTE karti hai
 *
 * Yeh wahi hai jo:
 *   - Mumbai mein ek laptop ko Virginia mein ek API server tak pahunchata hai
 *   - Ya ek cloud VPC mein ek service ko dusre region ke database subnet tak
 *     pahunchata hai
 *
 * Layer 3 IP ADDRESSES use karti hai:
 *   - Most commonly IPv4 aur IPv6
 *   - Routers DESTINATION IP address dekhte hain
 *   - Aur har packet ko ek hop CLOSER forward karte hain destination tak
 *
 * LAYER 3 RESPONSIBILITIES:
 *   1. IP addressing
 *   2. Routing between networks
 *   3. Packet forwarding
 *   4. Network reachability aur error reporting through ICMP
 *   5. Packet size discovery aur fragmentation behaviour
 *
 * Example:
 *   Network A: 192.168.1.0/24
 *     Client 192.168.1.10 -> "Packet to 10.0.0.5" -> Router (Routing Table)
 *     -> "Forward toward destination" -> Service 10.0.0.5
 *   Network B: 10.0.0.0/24
 *
 * ============================================================================
 *  MAC vs IP ADDRESSES (Fark Kya Hai)
 * ============================================================================
 *
 * MAC aur IP addresses ALAG PROBLEMS solve karte hain:
 *
 *   - MAC ADDRESS: Ek link pe LOCAL DELIVERY ke liye use hota hai
 *     (jaise aapke laptop se aapke router tak)
 *
 *   - IP ADDRESS: Networks ke across ROUTING ke liye use hota hai
 *
 * IMPORTANT:
 *   - Jab traffic aapka LOCAL NETWORK chhodta hai
 *   - Destination MAC address HAR HOP pe CHANGE hota hai
 *   - Destination IP usually SAME rehta hai jab tak packet target na pahunche
 *   - Ya koi device ise NAT ya proxying ke through REWRITE na kare
 *
 * ============================================================================
 *  PART 12: IP ADDRESSING IN BACKEND WORK
 * ============================================================================
 *
 * IP addressing BACKEND WORK mein CONSTANTLY dikhta hai:
 *
 *   1. VPC aur subnet design
 *   2. Kubernetes pod aur service networking
 *   3. Load balancer listeners aur target groups
 *   4. Firewall rules aur security groups
 *   5. Database allowlists
 *   6. DNS records
 *   7. NAT gateways aur egress controls
 *
 * ============================================================================
 *  IP PACKET ADDRESSES (Har Packet Mein Kam Se Kam 2 Addresses)
 * ============================================================================
 *
 *   1. SOURCE IP: Packet KAHAN SE AAYA dikh raha hai
 *
 *   2. DESTINATION IP: Packet kahan DELIVER hona chahiye
 *
 * Routers KYA KARTE HAIN:
 *   - Destination IP dekhte hain
 *   - Use apni ROUTING TABLE se COMPARE karte hain
 *   - Packet ko NEXT HOP pe forward karte hain
 *
 * Har router ko sirf NEXT STEP pata hona chahiye
 * - Ise CLIENT se SERVER tak POORI PATH jaanne ki zaroorat NAHI hai
 *
 * ============================================================================
 *  PRIVATE ADDRESSES (Private Addresses Kaise Kaam Karte Hain)
 * ============================================================================
 *
 * Private addresses ko sirf US NETWORK ke andar UNIQUE hona chahiye jahan
 * woh use hote hain
 *
 *   - Aapka laptop aur aapke PADOSI ka laptop DONO 192.168.1.10 ho sakte hain
 *   - Kyunki woh addresses ALAG ROUTERS ke peeche (behind) rehte hain
 *
 * ============================================================================
 *  CARRIER-GRADE NAT
 * ============================================================================
 *
 * Ek aur range ISP aur mobile networks mein AKSAR dikhti hai:
 *
 *   ┌────────────────────────────────┬────────────────────┬──────────────────────────────┐
 *   │ Range                          │ CIDR               │ Purpose                      │
 *   ├────────────────────────────────┼────────────────────┼──────────────────────────────┤
 *   │ 100.64.0.0 to 100.127.255.255  │ 100.64.0.0/10      │ Shared address space for     │
 *   │                                │                    │ carrier-grade NAT            │
 *   └────────────────────────────────┴────────────────────┴──────────────────────────────┘
 *
 * CARRIER-GRADE NAT kya karta hai:
 *   - Providers ko allow karta hai ki bahut saare customers ko CHHOTE number of
 *     public IPv4 addresses ke peeche rakh dein
 *   - Yeh IPv4 SCARCITY mein help karta hai
 *   - Lekin yeh INHEIN MUSHKIL banata hai:
 *     * Inbound connectivity
 *     * Abuse tracking
 *     * Kuch peer-to-peer protocols
 *
 * ============================================================================
 *  PART 13: NAT (Network Address Translation)
 * ============================================================================
 *
 * NAT ka full form: "Network Address Translation"
 * - NAT PACKET ADDRESSES ko REWRITE karta hai jab traffic ek NETWORK BOUNDARY
 *   cross karta hai
 *
 * SABSE COMMON FORM: SOURCE NAT for OUTBOUND traffic
 *
 * NAT KAISE KAAM KARTA HAI (Step by step):
 *
 *   1. Ek PRIVATE host internet pe ek packet bhejta hai
 *
 *   2. NAT device PRIVATE SOURCE IP ko PUBLIC IP se REPLACE karta hai
 *      - Aur us MAPPING ko RECORD karta hai
 *
 *   3. Jab RESPONSE wapas aata hai
 *   - NAT device US MAPPING ko use karta hai
 *   - Response ko SAHI INTERNAL HOST ko bhejne ke liye
 *
 * ============================================================================
 *  PART 14: ROUTING (Longest Prefix Match)
 * ============================================================================
 *
 * Routers LONGEST PREFIX MATCH use karte hain
 * - Matlab: SABSE SPECIFIC matching route JEETTA hai
 *
 * Example:
 *   Agar ek packet for 10.0.1.25 match karta hai:
 *     - 10.0.0.0/8  AUR
 *     - 10.0.1.0/24
 *   Toh /24 JEETTA hai kyunki yeh CHHOTA aur ZYADA SPECIFIC range describe
 *   karta hai
 *
 * ============================================================================
 *  HOP-BY-HOP FORWARDING
 * ============================================================================
 *
 * IP routing HOP BY HOP hai
 * - Ek router ko POORI PATH pata honi ki zaroorat NAHI hai
 * - Ise sirf BEST MATCHING ROUTE ke liye NEXT HOP pata hona chahiye
 *
 * PATH INSPECT karne ke TOOLS:
 *   - traceroute
 *   - tracepath
 *
 * WARNING - Inka output INCOMPLETE ya MISLEADING ho sakta hai:
 *   - Firewalls ke wajah se
 *   - ICMP filtering ke wajah se
 *   - Asymmetric routing ke wajah se
 *   - Load balancing ke wajah se
 *
 * Example traceroute output (documentation IP ranges use karta hai):
 *   $ traceroute api.example.com
 *   1  192.168.1.1   1.2 ms   local gateway
 *   2  10.0.0.1      5.8 ms   ISP gateway
 *   3  198.51.100.10 8.3 ms   provider network
 *   4  203.0.113.20  9.1 ms   edge network
 *   5  192.0.2.34    9.3 ms   destination
 *
 * NOTE: Yeh example DOCUMENTATION IP RANGES use karta hai
 * - Real traceroute output un addresses ko dikhayega jo AAPKE ISP, cloud
 *   provider, CDN, ya transit networks ke hain
 *
 * ============================================================================
 *  TTL AND HOP LIMIT
 * ============================================================================
 *
 * - IPv4 ek TTL (Time to Live) field use karta hai
 * - IPv6 ek HOP LIMIT field use karta hai
 * - Dono ka PURPOSE SAME hai: PACKETS KO FOREVER LOOP HONE SE ROKNA
 *
 * KAISE KAAM KARTA HAI:
 *   - Har router value ko 1 se DECREMENT karta hai
 *   - Jab value 0 ho jaati hai:
 *     * Router packet ko DROP kar deta hai
 *     * Usually ek ICMP TIME EXCEEDED message bhejta hai
 *
 * TTL FLOW DIAGRAM:
 *   Source (TTL=64) -> Decrement -> Router 1 (TTL=63) -> Decrement ->
 *   Router 2 (TTL=62) -> Decrement -> Router 3 (TTL=61) -> Deliver ->
 *   Destination (TTL=60)
 *
 * TRACEROUTE KAISE DISCOVER KARTA HAI:
 *   - Yeh INCREASING TTL ya HOP LIMIT values ke saath packets bhejta hai
 *   - Aur un ROUTERS ko RECORD karta hai jo EXPIRATION report karte hain
 *
 * ============================================================================
 *  BGP (Border Gateway Protocol)
 * ============================================================================
 *
 * INTERNET SCALE pe routing BGP se coordinate hota hai
 * - BGP ka full form: "Border Gateway Protocol"
 *
 * Internet BAHUT SAARE LARGE NETWORKS se bana hai:
 *   - ISPs
 *   - Cloud providers
 *   - CDNs
 *   - Enterprises
 *   - Aur others
 *
 * BGP un networks ko allow karta hai:
 *   1. ANNOUNCE karna ki woh KAUNSE IP RANGES tak pahunch sakte hain
 *   2. ROUTING POLICY ke according PATHS choose karna
 *
 * BGP POWERFUL hai lekin BLUNT hai:
 *   - Yeh NAHI jaanta ki aapka application HEALTHY hai ya nahi
 *   - Yeh sirf itna jaanta hai ki ek IP range EK PATH ke through REACHABLE
 *     dikh raha hai
 *
 * PROBLEMS jo ho sakti hain:
 *   - Route leaks
 *   - Bad announcements
 *   - Accidental withdrawals
 *   - Yeh BADE SERVICES ko UNREACHABLE bana sakte hain
 *
 * REAL EXAMPLE - October 2021 Facebook Outage:
 *   - Route changes ne Facebook, Instagram, aur WhatsApp ko GHANTO KE LIYE
 *     UNREACHABLE bana diya
 *   - Kyunki relevant network prefixes GLOBAL ROUTING se GAYAB ho gaye
 *   - Yeh HTTP problem NAHI tha
 *   - Yeh database problem NAHI tha
 *   - Yeh ROUTING LAYER pe REACHABILITY problem tha
 *
 * ============================================================================
 *  PART 15: IP ADDRESSES IN SYSTEM DESIGN
 * ============================================================================
 *
 * IP addressing choices SHAPE karte hain:
 *   - Reliability
 *   - Security
 *   - Day-to-day operations
 *
 * ============================================================================
 *  DO NOT TREAT IP AS IDENTITY (IP Ko Identity Mat Samjho!)
 * ============================================================================
 *
 * Ek IP address:
 *   - CHANGE ho sakta hai
 *   - SHARE ho sakta hai
 *   - TRANSLATE ho sakta hai
 *   - Ek PROXY, LOAD BALANCER, ya NAT GATEWAY represent kar sakta hai
 *
 * IMPORTANT:
 *   - IP address ko AKELA permanent user identity ya security decision ke roop
 *     mein USE MAT KARO
 *
 * COMMON PITFALLS (Aam Galtiyan):
 *   1. PUBLIC IP se RATE LIMITING:
 *      - Ek NAT ke peeche BAHUT SAARE unrelated users ko GROUP kar sakta hai
 *
 *   2. OFFICE IPs ko ALLOWLISTING:
 *      - Yeh BREAK ho jaata hai jab employees VPNs, mobile networks, ya
 *        cloud-hosted tools use karte hain
 *
 *   3. LOGS mein PROXY ya LOAD BALANCER IP dikhta hai:
 *      - Jab tak original client IP forwarding CORRECTLY CONFIGURED nahi hai
 *
 *   4. CONTAINERS aur PODS ke paas SHORT-LIVED ADDRESSES hote hain
 *
 * BEST PRACTICE:
 *   - IPs ko EK SIGNAL ke roop mein use karo
 *   - WHOLE IDENTITY MODEL mat banao
 *
 * ============================================================================
 *  PRESERVE CLIENT ADDRESS CAREFULLY
 * ============================================================================
 *
 * Applications ko aksar ORIGINAL CLIENT IP chahiye hota hai:
 *   - Abuse detection (galat use pakadna)
 *   - Geolocation (kahan se aa raha hai)
 *   - Audit logs (kaun aaya, kab aaya)
 *   - Rate limiting
 *
 * PROBLEM:
 *   - Jab traffic PROXIES ya LOAD BALANCERS se guzarta hai
 *   - Application ko jo source IP dikhta hai woh PROXY ka ho sakta hai
 *
 * COMMON MECHANISMS (Original IP Preserve Karne Ke Tarike):
 *   1. X-Forwarded-For
 *   2. Forwarded
 *   3. Proxy Protocol
 *   4. Cloud load balancer metadata
 *
 * SECURITY WARNING:
 *   - In values ko SIRF us infrastructure se TRUST karo jo AAP CONTROL karte ho
 *   - Ek CLIENT HTTP HEADERS FAKE kar sakta hai
 *   - Jab tak ek TRUSTED PROXY unhe pehle OVERWRITE ya CLEAN na kare
 *
 * ============================================================================
 *  PLAN FOR ADDRESS EXHAUSTION (Address Khatam Hone Ki Taiyari)
 * ============================================================================
 *
 * Good IP plans GROWTH ke liye JAGAH chhodte hain
 * - Yeh OVERLAP se bhi bachate hain un networks se jinhe aap baad mein connect
 *   kar sakte hain
 *
 * OVERLAPPING CIDR RANGES sabse MEHENGE CLOUD NETWORKING MISTAKES mein se ek
 * hain:
 *   - Agar do VPCs DONO 10.0.0.0/16 use karte hain
 *   - Toh unke beech routing MUSHKIL ya IMPOSSIBLE ho sakta hai
 *   - Bina NAT, proxying, renumbering, ya complex translation ke
 *
 * PRACTICAL GUIDANCE (Amal Mein Lao):
 *
 *   1. HAMESHA SAME DEFAULT RANGE use mat karo
 *
 *   2. FUTURE ENVIRONMENTS aur REGIONS ke liye SPACE RESERVE karo
 *
 *   3. Production, staging, data, aur shared services ko PREDICTABLE ranges
 *      mein rakho
 *
 *   4. Cloud CIDRs choose karne se PEHLE existing CORPORATE, VPN, PARTNER,
 *      aur DATA-CENTER ranges CHECK karo
 *
 *   5. Kubernetes POD IP USAGE EARLY mein ACCOUNT karo
 *      - Bahut saare pods wale clusters addresses EXPECTED SE TEZ consume kar
 *        sakte hain
 *
 *   6. Yaad rakho ki cloud subnets PROVIDER-SPECIFIC ADDRESSES reserve kar
 *      sakte hain
 *
 * IMPORTANT:
 *   - Ek ADDRESS PLAN = INFRASTRUCTURE ARCHITECTURE hai
 *   - Ise WAISE HI TREAT KARO
 *
 * ============================================================================
 *  PART 16: SUMMARY (Poore Document Ka Saransh)
 * ============================================================================
 *
 * TCP aur UDP TRANSPORT-LAYER TOOLS hain ALAG PROMISES ke saath:
 *
 *   1. TCP ek RELIABLE, ORDERED BYTE STREAM deta hai
 *      - Yeh bytes ko ORDER mein rakhta hai
 *      - Lekin yeh APPLICATION MESSAGE BOUNDARIES preserve NAHI karta
 *      - Aur yeh PROVE NAHI karta ki BUSINESS OPERATION SUCCEEDED
 *
 *   2. UDP INDEPENDENT DATAGRAMS deta hai
 *      - Yeh DATAGRAM BOUNDARIES preserve karta hai
 *      - Lekin yeh DELIVERY ya ORDER ka PROMISE NAHI deta
 *
 *   3. TCP mein FLOW CONTROL aur CONGESTION CONTROL included hai
 *      - UDP applications ko apni PACING, RELIABILITY, aur RECOVERY khud
 *        add karni padti hai jab unhe in features ki zaroorat ho
 *
 *   4. QUIC UDP ke upar chalta hai aur add karta hai:
 *      - Encryption
 *      - Streams
 *      - Recovery
 *      - Congestion control
 *      - Connection migration
 *
 * BEST DESIGN QUESTION:
 *   "Kaunsa TEZ hai?" - YEH NAHI poochna chahiye
 *
 * ASLI SAWAAL yeh hai:
 *   "Application ko KYA KARNA CHAHIYE jab data LATE ho, LOST ho,
 *    DUPLICATED ho, REORDERED ho, ya DO BAAR PROCESSED ho?"
 *
 * ============================================================================
 **/